In [1]:
!pip install -q langchain-core langchain-text-splitters langchain-huggingface langchain-chroma chromadb "langchain-docling[local]" langsmith

In [2]:
from langchain_docling import DoclingLoader

pdf_path = "/kaggle/input/datasets/sairajanvekar/labtest-dataset/budget_speech.pdf"

loader = DoclingLoader(file_path=pdf_path)
documents = loader.load()

print("Number of extracted documents:", len(documents))

print("\nFirst extracted text:\n")
print(documents[0].page_content[:2500])

print("\nMetadata:\n")
print(documents[0].metadata)

The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
[WARNING] 2026-11-13 13:48:43,255 [RapidOCR] provider_config.py:103: CUDAExecutionProvider is not in available providers (['AzureExecutionProvider', 'CPUExecutionProvider']). Use AzureExecutionProvider inference by default.
[WARNING] 2026-11-13 13:48:43,328 [RapidOCR] provider_config.py:103: CUDAExecutionProvider is not in available providers (['AzureExecutionProvider', 'CPUExecutionProvider']). Use AzureExecutionProvider inference by default.
[WARNING] 2026-11-13 13:48:44,207 [RapidOCR] provider_config.py:103: CUDAExecutionProvider is not in available providers (['AzureExecutionProvider', 'CPUExecutionProvider']). Use AzureExecutionProvider inference by default.
[WARNING] 2026-11-13 13:48:44,280 [RapidOCR] provider_config.py:103: CUDAExecutionProvider is not in available providers (['AzureExecutionProvider', 'CPUExecutionProvider']). Use AzureExecutionProvider inference

2026-11-13 13:48:48,170 MatchingPostProcessor WARNING  1 of 295 pdf cells matched neither a row nor a column band of the 8x3 grid and were dropped from the table


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (566 > 512). Running this sequence through the model will result in indexing errors


Number of extracted documents: 171

First extracted text:

SPEECH OF NIRMALA SITHARAMAN MINISTER OF FINANCE
February 1, 2026

Metadata:

{'source': '/kaggle/input/datasets/sairajanvekar/labtest-dataset/budget_speech.pdf', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/3', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 1, 'bbox': {'l': 224.4, 't': 164.22384, 'r': 332.07948995799995, 'b': 151.79327999999998, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 16]}]}], 'headings': ['SPEECH OF NIRMALA SITHARAMAN MINISTER OF FINANCE'], 'origin': {'mimetype': 'application/pdf', 'binary_hash': 5207691758262864124, 'filename': 'budget_speech.pdf'}}}


In [3]:
for i in range(min(10, len(documents))):
    doc = documents[i]

    print(f"\n--- Document {i + 1} ---")
    print(doc.page_content[:500])

    # Extract page numbers from Docling metadata
    page_numbers = []

    for item in doc.metadata.get("dl_meta", {}).get("doc_items", []):
        for prov in item.get("prov", []):
            page_numbers.append(prov.get("page_no"))

    print("Page numbers:", sorted(set(p for p in page_numbers if p is not None)))


--- Document 1 ---
SPEECH OF NIRMALA SITHARAMAN MINISTER OF FINANCE
February 1, 2026
Page numbers: [1]

--- Document 2 ---
CONTENTS
| - |
| Yuva Shakti and 3 kartavya, Page No. 1.PART - A = 2. Reform Express, Page No. 1.PART - A = 3. First kartavya: to accelerate and sustain economic growth, Page No. 1.PART - A = 3. Second kartavya: fulfil aspirations and build capacity, Page No. 1.PART - A = 10. Third kartavya: Sabka Sath, Sabka Vikas, Page No. 1.PART - A = 14. 16 th Finance Commission, Page No. 1.PART - A = 18. Fiscal Consolidation, Page No. 1.PART - A = 18. PART - B, Page No. 1.PART - A = PART - B. Direct taxes, 
Page numbers: [3]

--- Document 3 ---
CONTENTS
| - |
 1.PART - A = 50 |

Page numbers: [3]

--- Document 4 ---
February 1, 2026
Hon'ble Speaker, On the sacred occasion of Magha Purnima and the birth anniversary of Guru Ravidas, I present the Budget for the year 2026-2027.
Page numbers: [5]

--- Document 5 ---
Introduction
1. Since we assumed office 12 years ago, India's ec

In [4]:
import re
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

cleaned_documents = []

for doc in documents:
    text = doc.page_content

    if "CONTENTS" in text[:100]:
        text = re.sub(r"\|\s*-+\s*\|", "", text)
        text = re.sub(r"\s*\|\s*", " ", text)
        text = re.sub(r"Page No\.\s*1\.PART", "Page No. 1. PART", text)

    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()

    pages = [
        prov["page_no"]
        for item in doc.metadata.get("dl_meta", {}).get("doc_items", [])
        for prov in item.get("prov", [])
        if prov.get("page_no") is not None
    ]

    metadata = {
        "source": doc.metadata.get("source", pdf_path),
        "page": min(pages) if pages else -1,
        "pages": ",".join(map(str, sorted(set(pages))))
    }

    cleaned_documents.append(Document(page_content=text, metadata=metadata))

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

texts = splitter.split_documents(cleaned_documents)

print("Original documents:", len(documents))
print("Cleaned documents:", len(cleaned_documents))
print("Total chunks:", len(texts))
print("\nSample chunk:\n", texts[0].page_content)
print("\nSample metadata:\n", texts[0].metadata)

Original documents: 171
Cleaned documents: 171
Total chunks: 213

Sample chunk:
 SPEECH OF NIRMALA SITHARAMAN MINISTER OF FINANCE
February 1, 2026

Sample metadata:
 {'source': '/kaggle/input/datasets/sairajanvekar/labtest-dataset/budget_speech.pdf', 'page': 1, 'pages': '1'}


In [5]:
for i, chunk in enumerate(texts[:5]):
    print(f"\nChunk {i + 1}")
    print("Page:", chunk.metadata["page"])
    print("Pages:", chunk.metadata["pages"])
    print("Text:", chunk.page_content[:150])


Chunk 1
Page: 1
Pages: 1
Text: SPEECH OF NIRMALA SITHARAMAN MINISTER OF FINANCE
February 1, 2026

Chunk 2
Page: 3
Pages: 3
Text: CONTENTS Yuva Shakti and 3 kartavya, Page No. 1. PART - A = 2. Reform Express, Page No. 1. PART - A = 3. First kartavya: to accelerate and sustain eco

Chunk 3
Page: 3
Pages: 3
Text: CONTENTS

 1.PART - A = 50

Chunk 4
Page: 5
Pages: 5
Text: February 1, 2026
Hon'ble Speaker, On the sacred occasion of Magha Purnima and the birth anniversary of Guru Ravidas, I present the Budget for the year

Chunk 5
Page: 5
Pages: 5,6
Text: Introduction
1. Since we assumed office 12 years ago, India's economic trajectory has been marked by stability, fiscal discipline, sustained growth an


------------------------

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

def clean_metadata(doc):
    doc.metadata = {
        k: v for k, v in doc.metadata.items()
        if isinstance(v, (str, int, float, bool))
    }
    return doc

clean_texts = [clean_metadata(doc) for doc in texts]

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

persist_directory = "/kaggle/working/budget_chroma"

vectorstore = Chroma.from_documents(
    documents=clean_texts,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name="budget_speech"
)

print("Vector database created.")
print("Chunks indexed:", len(clean_texts))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Vector database created.
Chunks indexed: 213


In [7]:
reloaded_vectorstore = Chroma(
    collection_name="budget_speech",
    embedding_function=embeddings,
    persist_directory=persist_directory
)

print("Chroma vector store reloaded successfully.")
print("Persisted database location:", persist_directory)

Chroma vector store reloaded successfully.
Persisted database location: /kaggle/working/budget_chroma


In [8]:
query = "Biopharma SHAKTI outlay"

results = reloaded_vectorstore.similarity_search_with_score(
    query,
    k=3
)

print("Query:", query)
print("\nTop 3 search results:\n")

for rank, (doc, score) in enumerate(results, start=1):
    print(f"--- Result {rank} ---")
    print("Distance score:", round(score, 4))
    print("Page:", doc.metadata.get("page", "Unknown"))
    print("Text:", doc.page_content[:500])
    print()

Query: Biopharma SHAKTI outlay

Top 3 search results:

--- Result 1 ---
Distance score: 0.5278
Page: 8
Text: 16. Biopharma SHAKTI (Strategy for Healthcare Advancement through Knowledge, Technology and Innovation) - India's disease burden is observed to be shifting towards non-communicable diseases, like diabetes, cancer and autoimmune disorders. Biologic medicines are key to longevity and quality of life at affordable costs. To develop India as a global Biopharma manufacturing hub, I propose the Biopharma SHAKTI with an outlay of ₹ 10,000 crores over the next 5 years. This will build the ecosystem for 

--- Result 2 ---
Distance score: 0.5278
Page: 8
Text: 16. Biopharma SHAKTI (Strategy for Healthcare Advancement through Knowledge, Technology and Innovation) - India's disease burden is observed to be shifting towards non-communicable diseases, like diabetes, cancer and autoimmune disorders. Biologic medicines are key to longevity and quality of life at affordable costs. To develop Indi

----------------------------

In [9]:
!pip install -q langchain-groq

In [10]:
from kaggle_secrets import UserSecretsClient
from langchain_groq import ChatGroq

secrets = UserSecretsClient()
groq_api_key = secrets.get_secret("GROQ_API_KEY")

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=groq_api_key,
)

print("ChatGroq initialized successfully.")

ChatGroq initialized successfully.


In [11]:
from langchain_core.tools import tool

@tool
def search_budget(query: str) -> str:
    """Search the budget speech and return relevant text with page numbers."""
    results = reloaded_vectorstore.similarity_search_with_score(query, k=3)

    return "\n\n".join(
        f"Page: {doc.metadata.get('page', 'Unknown')}\n"
        f"Score: {score:.4f}\n"
        f"Text: {doc.page_content}"
        for doc, score in results
    )

print("Document-search tool created successfully.")

Document-search tool created successfully.


In [12]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", """You answer questions about the Union Budget 2026-27.
    Use search_budget for budget-specific facts and calculator for arithmetic.
    Answer greetings and general questions without tools.
    Cite PDF page numbers for budget-related answers.
    If information is missing, say "No relevant information found in the attached source."
    Do not guess or repeat the same tool call unnecessarily.
    Once you have enough information, give a concise final answer."""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])


In [13]:
from langchain_core.messages import HumanMessage, ToolMessage

tool_map = {"search_budget": search_budget}
chat_history = []

llm_with_tools = llm.bind_tools(list(tool_map.values()))

def run_agent(user_input, max_rounds=5):
    messages = prompt.format_messages(
        input=user_input,
        chat_history=chat_history
    )

    for _ in range(max_rounds):
        response = llm_with_tools.invoke(messages)

        if not response.tool_calls:
            chat_history.extend([
                HumanMessage(content=user_input),
                response
            ])
            return response.content

        messages.append(response)

        for call in response.tool_calls:
            name = call["name"]
            print(f"Tool used: {name}")

            result = (
                tool_map[name].invoke(call["args"])
                if name in tool_map
                else "Unknown tool."
            )

            messages.append(
                ToolMessage(
                    content=str(result),
                    tool_call_id=call["id"]
                )
            )

    return "Could not produce a final answer within the tool-call limit."

---------------

In [14]:
from langchain_core.tools import tool
import ast
import operator

ops = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
}

def calculate_node(node):
    if isinstance(node, ast.Constant) and type(node.value) in (int, float):
        return node.value

    if isinstance(node, ast.BinOp) and type(node.op) in ops:
        return ops[type(node.op)](
            calculate_node(node.left),
            calculate_node(node.right)
        )

    raise ValueError("Invalid arithmetic expression")

@tool
def calculator(expression: str) -> str:
    """Calculate arithmetic expressions and percentage changes."""
    try:
        tree = ast.parse(expression, mode="eval")
        return str(calculate_node(tree.body))
    except Exception as e:
        return f"Calculation error: {e}"

print(calculator.invoke({
    "expression": "((12.2 - 11.2) / 11.2) * 100"
}))

8.928571428571429


In [15]:
from langchain_core.messages import HumanMessage, ToolMessage

tool_map = {
    "search_budget": search_budget,
    "calculator": calculator,
}

llm_with_tools = llm.bind_tools(list(tool_map.values()))
chat_history = []

def run_agent(user_input, max_rounds=8):
    messages = prompt.format_messages(
        input=user_input,
        chat_history=chat_history
    )
    tools_used = []

    for _ in range(max_rounds):
        response = llm_with_tools.invoke(messages)

        if not response.tool_calls:
            chat_history.extend([
                HumanMessage(content=user_input),
                response
            ])
            return {
                "answer": response.content,
                "tools_used": tools_used,
                "searched": "Y" if "search_budget" in tools_used else "N"
            }

        messages.append(response)

        for call in response.tool_calls:
            name = call["name"]
            tools_used.append(name)
            print(f"Tool used: {name}")

            try:
                result = tool_map[name].invoke(call["args"])
            except Exception as e:
                result = f"Tool error: {e}"

            messages.append(
                ToolMessage(
                    content=str(result),
                    tool_call_id=call["id"]
                )
            )

    return {
        "answer": "Could not produce a final answer within the tool-call limit.",
        "tools_used": tools_used,
        "searched": "Y" if "search_budget" in tools_used else "N"
    }

print("Agent ready")

Agent ready


---------------------------

In [16]:
import pandas as pd
from IPython.display import display

chat_history = []
test_results = []

while True:
    user_input = input("\nYou: ").strip()

    if user_input.lower() == "exit":
        print("Chat ended.")
        break

    if not user_input:
        continue

    result = run_agent(user_input)
    print("\nAssistant:", result["answer"])

    test_results.append({
        "Query": user_input,
        "Searched (Y/N)": result["searched"],
        "Tools used": ", ".join(result["tools_used"]) or "None",
        "Final answer": result["answer"]
    })

results_df = pd.DataFrame(test_results)
print("\nFINAL TEST RESULTS")
display(results_df)


You:  What is the outlay of Biopharma SHAKTI?


Tool used: search_budget

Assistant: The outlay proposed for the Biopharma SHAKTI initiative is **₹10,000 crores** over the next five years【search_budget 8】.



You:  How does the 2026-27 fiscal deficit compare with 2025-26?


Tool used: search_budget

Assistant: **Fiscal deficit comparison**

| Fiscal year | Estimated deficit (as % of GDP) |
|-------------|---------------------------------|
| 2025‑26 (BE) | **4.4 %** |
| 2026‑27 (BE) | **4.3 %** |

The 2026‑27 fiscal deficit is projected to be slightly lower than that of 2025‑26, dropping from 4.4 % to 4.3 % of GDP【search_budget 8】.



You:  What changes for NCCD on chewing tobacco, and what is the effective duty?


Tool used: search_budget

Assistant: **NCCD on chewing tobacco (2026‑27)**  

- **Rate change:** The NCCD rate is **raised from 25 % to 60 %**【search_budget 8】.  
- **Effective duty:** The effective duty rate remains **25 %** by notification【search_budget 8】.


... (9 outputs left out)

Tool used: search_budget
Tool used: search_budget
Tool used: search_budget
Tool used: search_budget
Tool used: search_budget
Tool used: search_budget
Tool used: search_budget

Assistant: No relevant information found in the attached source.



You:  exit


Chat ended.

FINAL TEST RESULTS


,Query,Searched (Y/N),Tools used,Final answer
0,What is the outlay of Biopharma SHAKTI?,Y,search_budget,The outlay proposed for the Biopharma SHAKTI i...
1,How does the 2026-27 fiscal deficit compare wi...,Y,search_budget,**Fiscal deficit comparison**\n\n| Fiscal year...
2,"What changes for NCCD on chewing tobacco, and ...",Y,search_budget,**NCCD on chewing tobacco (2026‑27)** \n\n- *...
3,"Hi, how are you?",N,None,Hello! I'm doing great—thanks for asking. How ...
4,Explain fiscal deficit in simple words,N,None,A fiscal deficit happens when a government spe...
5,What is the combined outlay of Biopharma SHAKT...,Y,"search_budget, search_budget, search_budget",The combined budgetary outlay for:\n\n- **Biop...
6,By what percentage does public capex rise from...,Y,"search_budget, calculator",Public capex rises from ₹11.2 lakh crore (BE 2...
7,What are the new income tax slab rates in this...,Y,"search_budget, search_budget, search_budget, s...",No relevant information found in the attached ...
